In [1]:
# 4_feature_eng_ukhls.ipynb
#
# Applies feature engineering to the backfilled UKHLS Wave O data.
# All operations are driven by config_variables.py — no hardcoded lists.
#
# Steps:
#   1. Load o_indresp_backfilled.pkl
#   2. Apply value recodes (RECODE_MAPS)     e.g. hiqual_dv 9 → 5
#   3. Apply floor clipping  (FLOOR_VALUES)  e.g. payn_dv < 0 → 0
#   4. Apply upper clipping  (CLIP_VALUES)   e.g. carmiles > 50,000 → 50,000
#   5. One-hot encode categorical variables  (ONE_HOT_VARS) — originals kept
#   6. Force ALL columns to float32 (includes OHE columns added in step 5)
#   7. Save as o_indresp_feature_eng.pkl

import sys, os
sys.path.insert(0, os.path.abspath('..'))

import importlib
import data_processing.config_variables as _ukhls_vars
importlib.reload(_ukhls_vars)

import pandas as pd
import numpy as np

from data_processing.config_variables import (
    RECODE_MAPS,
    FLOOR_VALUES,
    CLIP_VALUES,
    ONE_HOT_VARS,
    VARIABLES,
)

# ── Config ────────────────────────────────────────────────────────────────────
WAVE       = "o"
INPUT_PKL  = "../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl"
OUTPUT_PKL = "../data/4_feature_eng_ukhls/o_indresp_feature_eng.pkl"

def get_base_code(col_name, wave_prefix):
    prefix = f"{wave_prefix}_"
    return col_name[len(prefix):] if col_name.startswith(prefix) else col_name

# ── 1. Load ───────────────────────────────────────────────────────────────────
print(f"Loading {INPUT_PKL} ...")
df = pd.read_pickle(INPUT_PKL)
print(f"Loaded {len(df):,} rows × {len(df.columns)} columns")

# ── 2. Value Recodes ──────────────────────────────────────────────────────────
print("\nStep 2: Applying value recodes ...")
recoded = []
for col in df.columns:
    base = get_base_code(col, WAVE)
    recode = RECODE_MAPS.get(base)
    if recode:
        df[col] = pd.to_numeric(df[col], errors='coerce').replace(recode)
        recoded.append(f"  {col}: {recode}")
if recoded:
    print("\n".join(recoded))
else:
    print("  (none)")

# ── 3. Floor Clipping ─────────────────────────────────────────────────────────
print("\nStep 3: Applying floor values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    floor_val = FLOOR_VALUES.get(base)
    if floor_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_floored = int((numeric < floor_val).sum())
        df[col] = numeric.clip(lower=floor_val)
        print(f"  {col}: floored at {floor_val}  ({n_floored:,} rows affected)")

# ── 4. Upper Clipping ─────────────────────────────────────────────────────────
print("\nStep 4: Applying clip values ...")
for col in df.columns:
    base = get_base_code(col, WAVE)
    clip_val = CLIP_VALUES.get(base)
    if clip_val is not None:
        numeric = pd.to_numeric(df[col], errors='coerce')
        n_clipped = int((numeric > clip_val).sum())
        df[col] = numeric.clip(upper=clip_val)
        print(f"  {col}: clipped at {clip_val:,}  ({n_clipped:,} rows affected)")

# ── 5. One-hot encoding ───────────────────────────────────────────────────────
# For each variable with one_hot defined, create binary indicator columns (bool).
# Original column is kept unchanged.
# Naming: {wave}_{base}_{int(code)}  e.g. o_jbstat_2 = 1 if employed
# Type casting to float32 is handled centrally in step 6.
print("\nStep 5: One-hot encoding ...")
oh_new_cols = []
for base, one_hot_spec in ONE_HOT_VARS.items():
    col = f"{WAVE}_{base}"
    if col not in df.columns:
        print(f"  SKIP {col} — not in dataframe")
        continue

    src = pd.to_numeric(df[col], errors='coerce')

    # Determine which codes to encode
    if one_hot_spec is True:
        # Encode all categories defined in VARIABLES
        codes = list(VARIABLES[base]["categories"].keys())
    else:
        # one_hot_spec is a list of specific codes
        codes = list(one_hot_spec)

    for code in codes:
        new_col = f"{WAVE}_{base}_{int(code)}"
        df[new_col] = (src == code)   # bool for now; step 6 casts to float32
        oh_new_cols.append(new_col)
        print(f"  {new_col}  ({int((df[new_col]).sum()):,} = 1)")

print(f"\n  {len(oh_new_cols)} binary indicator columns added; originals kept.")

# ── 6. Force ALL columns to float32 ──────────────────────────────────────────
# Runs after step 5, so OHE columns added above are included.
print("\nStep 6: Converting all features to float32 ...")
for col in df.columns:
    if col == 'pidp':
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.int64)
    else:
        df[col] = pd.to_numeric(df[col], errors='coerce').astype(np.float32)

n_nulls = df.drop(columns=['pidp']).isna().sum().sum()
if n_nulls > 0:
    print(f"  WARNING — {n_nulls:,} NaN values remain after feature engineering")
    print(df.drop(columns=['pidp']).isna().sum()[lambda s: s > 0])
else:
    print("  No NaN values — clean feature matrix")

# ── 7. Save ───────────────────────────────────────────────────────────────────
df.to_pickle(OUTPUT_PKL, protocol=5)
print(f"\nDone. Feature matrix saved to {OUTPUT_PKL}")
print(f"Shape: {df.shape}")
print(df.dtypes)


Loading ../data/3_backfill_ukhls_waves/o_indresp_backfilled.pkl ...
Loaded 47,354 rows × 34 columns

Step 2: Applying value recodes ...
  o_hiqual_dv: {9.0: 5.0}

Step 3: Applying floor values ...
  o_payn_dv: floored at 0  (26,909 rows affected)

Step 4: Applying clip values ...
  o_jbttwt: clipped at 120  (41 rows affected)
  o_carmiles: clipped at 50,000  (67 rows affected)
  o_fimngrs_dv: clipped at 8,000  (1,040 rows affected)
  o_payn_dv: clipped at 8,000  (54 rows affected)
  o_nchild_dv: clipped at 7  (2 rows affected)

Step 5: One-hot encoding ...
  o_sex_dv_1  (21,482 = 1)
  o_englang_1  (7,817 = 1)
  o_jbstat_1  (3,282 = 1)
  o_jbstat_2  (22,247 = 1)
  o_jbstat_3  (2,210 = 1)
  o_jbstat_4  (12,778 = 1)
  o_jbstat_5  (266 = 1)
  o_jbstat_6  (1,251 = 1)
  o_jbstat_7  (2,555 = 1)
  o_jbstat_8  (1,792 = 1)
  o_jbstat_9  (12 = 1)
  o_jbstat_10  (23 = 1)
  o_jbstat_11  (152 = 1)

  13 binary indicator columns added; originals kept.

Step 6: Converting all features to float32 ...
 